# Building Shor's Algorithm from Scratch, Part 4: Modular Exponentiation

[Part 3](403_shor_scratch_controlled_multiplier.ipynb) built controlled modular multiplication: $\ket{\text{ctrl}}\ket{x}\ket{0} \to \ket{\text{ctrl}}\ket{x}\ket{\text{ctrl}\cdot(a x \bmod N)}$. This part builds the operation Shor's algorithm's quantum part actually runs: **modular exponentiation**,

$$
\ket{j}\ket{1} \longrightarrow \ket{j}\ket{a^j \bmod N}
$$

— computing $a^j \bmod N$ *in superposition* over every value of $j$ the phase-estimation register holds at once. This is the last arithmetic building block in this series; see [400_shor.ipynb](400_shor.ipynb) for how the surrounding phase estimation (Hadamards, this circuit, inverse QFT, measurement, continued fractions) turns it into an order-finder.

Reference: V. Vedral, A. Barenco, A. Ekert, ["Quantum Networks for Elementary Arithmetic Operations"](https://arxiv.org/abs/quant-ph/9511018) (1995).

## Why Part 3's multiplier isn't quite enough yet

$a^j \bmod N$ for a multi-bit $j = j_0 + 2j_1 + 4j_2 + \cdots$ is $a^{j_0} \cdot (a^2)^{j_1} \cdot (a^4)^{j_2} \cdots \bmod N$ — a product of repeated squarings, each included only if the corresponding bit of $j$ is 1. So the natural plan is: start a register at $1$, and for each bit $k$ of $j$, multiply it by $a^{2^k} \bmod N$ *controlled by that bit*.

But Part 3's `build_controlled_multiplier` computes $b \mathrel{+}= a \cdot x \bmod N$ **into a separate, fresh register** $b$ — it doesn't turn $x$ itself into $ax \bmod N$. That's fine for a single multiplication, but exponentiation needs to feed the *output* of one step into the *input* of the next, repeatedly, using the same register. We need a version that works **in place**: $x \to ax \bmod N$, with no separate output register left behind.

In [1]:
!pip install git+https://github.com/blueqat/blueqatSDK

/Users/yuichirominato/.pyenv/shims/pip: line 8: /opt/homebrew/opt/pyenv/bin/pyenv: No such file or directory


## In-place controlled multiplication: multiply, swap, un-multiply

The standard trick (same papers as Parts 1-3) needs one more ingredient: the modular inverse of $a$, i.e. $a^{-1}$ such that $a \cdot a^{-1} \equiv 1 \pmod N$ — which exists whenever $a$ and $N$ are coprime (Shor's algorithm always picks such an $a$; Python's `pow(a, -1, N)` computes it directly).

1. Run Part 3's multiplier: $\ket{x}\ket{0} \to \ket{x}\ket{ax \bmod N}$ (controlled).
2. **Swap** the $x$ register with the (now populated) result register, controlled by the same control qubit: $\ket{x}\ket{ax \bmod N} \to \ket{ax \bmod N}\ket{x}$.
3. Run Part 3's multiplier *backwards*, using $a^{-1}$ instead of $a$: since the second register now holds the *original* $x$, multiplying it by $a^{-1}$ and subtracting (i.e. running the whole circuit in reverse) turns it back into $\ket 0$ — because $a^{-1} \cdot x$ un-does exactly what putting $x$ there was going to compute, leaving a clean $\ket 0$ behind, uncomputed.

End state: $\ket{ax \bmod N}\ket{0}$ — the roles of the two registers have swapped, but the second one is clean again, so relabeling which register is "$x$" gives exactly the in-place update we wanted.

In [2]:
from blueqat import Circuit

def mcx(controls, target, ancillas):
    """X on `target`, controlled by ALL qubits in `controls` (see Part 3 for the derivation)."""
    circ = Circuit()
    n = len(controls)
    if n == 0:
        circ.x[target]
        return circ
    if n == 1:
        circ.cx[controls[0], target]
        return circ
    if n == 2:
        circ.ccx[controls[0], controls[1], target]
        return circ
    chain = []
    circ.ccx[controls[0], controls[1], ancillas[0]]
    chain.append(ancillas[0])
    for i in range(2, n - 1):
        circ.ccx[chain[-1], controls[i], ancillas[len(chain)]]
        chain.append(ancillas[len(chain)])
    circ.ccx[chain[-1], controls[-1], target]
    for i in range(len(chain) - 1, 0, -1):
        circ.ccx[chain[i - 1], controls[i + 1], chain[i]]
    circ.ccx[controls[0], controls[1], chain[0]]
    return circ

def carry_ops(c, a, b, c_next):
    return [((a, b), c_next), ((a,), b), ((c, b), c_next)]

def carry_inv_ops(c, a, b, c_next):
    return [((c, b), c_next), ((a,), b), ((a, b), c_next)]

def sum_ops(c, a, b):
    return [((a,), b), ((c,), b)]

def add_ops(c, a, b, n):
    ops = []
    for i in range(n - 1):
        ops += carry_ops(c[i], a[i], b[i], c[i + 1])
    ops += carry_ops(c[n - 1], a[n - 1], b[n - 1], b[n])
    ops.append(((a[n - 1],), b[n - 1]))
    ops += sum_ops(c[n - 1], a[n - 1], b[n - 1])
    for i in range(n - 2, -1, -1):
        ops += carry_inv_ops(c[i], a[i], b[i], c[i + 1])
        ops += sum_ops(c[i], a[i], b[i])
    return ops

def sub_ops(c, a, b, n):
    return list(reversed(add_ops(c, a, b, n)))

def modulo_add_ops(c, a, b, N, t, n):
    """b := (a + b) mod N -- the five-step construction from Part 2, as (controls, target) pairs."""
    ops = []
    ops += add_ops(c, a, b, n)
    ops += sub_ops(c, N, b, n)
    ops.append(((b[n],), t))
    ops += [(ctrl + (t,), tgt) for ctrl, tgt in add_ops(c, N, b, n)]
    ops += sub_ops(c, a, b, n)
    ops.append(((), b[n]))
    ops.append(((b[n],), t))
    ops.append(((), b[n]))
    ops += add_ops(c, a, b, n)
    return ops

def controlled_swap_ops(ctrl, a_reg, b_reg):
    """Swap a_reg[i] <-> b_reg[i] for every i, controlled by `ctrl` (Fredkin gate, built from CX/CCX/CX)."""
    ops = []
    for a, b in zip(a_reg, b_reg):
        ops.append(((b,), a))
        ops.append(((a, ctrl), b))
        ops.append(((b,), a))
    return ops

## The in-place controlled multiplier

Same constant-loading trick as Part 3 (the classical constants $a \cdot 2^i \bmod N$ and $a^{-1} \cdot 2^i \bmod N$ are known at circuit-build time), now with the swap and the reversed second pass added.

In [3]:
def build_controlled_Ua(mult_a, val_x, val_N, ctrl_value, n, n_ancilla=4):
    """x <- (a * x) mod N if ctrl else x, done truly in place."""
    c = list(range(n))                                  # carry register
    const = list(range(n, 2 * n))                        # scratch: holds a constant, loaded/unloaded via X
    res = list(range(2 * n, 3 * n + 1))                  # result register (n+1 wide, becomes the new x)
    N = list(range(3 * n + 1, 4 * n + 1))                 # modulus
    x = list(range(4 * n + 1, 5 * n + 1))                 # the value being exponentiated
    ctrl = 5 * n + 1
    t = 5 * n + 2                                         # modulo-adder overflow flag
    ancillas = list(range(5 * n + 3, 5 * n + 3 + n_ancilla))
    circ = Circuit(5 * n + 3 + n_ancilla)

    if ctrl_value:
        circ.x[ctrl]
    for i in range(n):
        if (val_x >> i) & 1:
            circ.x[x[i]]
        if (val_N >> i) & 1:
            circ.x[N[i]]

    def apply_modulo_add(constant, ops_source):
        nonlocal circ
        for i in range(n):
            const_val = (constant * (1 << i)) % val_N
            for j in range(n):
                if (const_val >> j) & 1:
                    circ.x[const[j]]
            ops = ops_source(c, const, res, N, t, n)
            for controls, target in ops:
                circ += mcx(list(controls) + [ctrl, x[i]], target, ancillas)
            for j in range(n):
                if (const_val >> j) & 1:
                    circ.x[const[j]]

    # Step A: res += a * x  mod N  (controlled)
    apply_modulo_add(mult_a, modulo_add_ops)
    # Step B: swap x <-> res[0..n-1] (controlled)
    for controls, target in controlled_swap_ops(ctrl, x, res[:n]):
        circ += mcx(list(controls), target, ancillas)
    # Step C: res -= a^-1 * x  mod N  (controlled) -- x now holds the old res value; this zeroes res back out
    a_inv = pow(mult_a, -1, val_N)
    apply_modulo_add(a_inv, lambda *args: list(reversed(modulo_add_ops(*args))))

    return circ, x, res

Quick check with Part 3's own numbers: $2 \times 1 \bmod 3 = 2$, and the scratch register should return to exactly $0$.

In [4]:
def run_Ua(mult_a, val_x, val_N, ctrl_value, n):
    circuit, x, res = build_controlled_Ua(mult_a, val_x, val_N, ctrl_value, n)
    total = circuit.n_qubits
    state = circuit.m[:].run(shots=1).most_common(1)[0][0]
    x_val = int("".join(reversed([state[total - 1 - q] for q in x])), 2)
    res_val = int("".join(reversed([state[total - 1 - q] for q in res])), 2)
    return x_val, res_val

for a, xv, N, ctrl in [(2, 1, 3, 1), (2, 2, 3, 1), (2, 1, 3, 0), (2, 0, 3, 1)]:
    x_val, res_val = run_Ua(a, xv, N, ctrl, 2)
    expected_x = (a * xv) % N if ctrl else xv
    ok = x_val == expected_x and res_val == 0
    print(f"ctrl={ctrl}: x={xv} -> {x_val} (expected {expected_x}), scratch={res_val} (expected 0)  {'OK' if ok else 'MISMATCH'}")

ctrl=1: x=1 -> 2 (expected 2), scratch=0 (expected 0)  OK


ctrl=1: x=2 -> 1 (expected 1), scratch=0 (expected 0)  OK


ctrl=0: x=1 -> 1 (expected 1), scratch=0 (expected 0)  OK


ctrl=1: x=0 -> 0 (expected 0), scratch=0 (expected 0)  OK


All four check out, including `ctrl=0` (x unchanged) and `x=0` (stays 0 — $a \cdot 0 = 0$ either way).

## Chaining it into full exponentiation

$j$'s bits, from least to most significant, control multiplication by $a^1, a^2, a^4, a^8, \ldots \bmod N$ in turn (each is just `pow(a, 2**k, N)`, computed classically once per bit, exactly like the per-bit constants inside the multiplier itself). $x$ starts at $1$ ($a^0$), and each step feeds the previous step's output back in as the next step's input.

In [5]:
def build_modexp(mult_a, val_N, exponent_val, n, n_exp_bits, n_ancilla=4):
    """x ends up holding a**exponent_val mod N. exponent_val is set classically here (via X gates) so we
    can check correctness against Python's pow(); in the real algorithm this register is a superposition
    prepared by Hadamards, and the whole circuit below runs unchanged on every branch at once."""
    c = list(range(n))
    const = list(range(n, 2 * n))
    res = list(range(2 * n, 3 * n + 1))
    N = list(range(3 * n + 1, 4 * n + 1))
    x = list(range(4 * n + 1, 5 * n + 1))
    j = list(range(5 * n + 1, 5 * n + 1 + n_exp_bits))       # exponent register, one control qubit per bit
    t = 5 * n + 1 + n_exp_bits
    ancillas = list(range(5 * n + 2 + n_exp_bits, 5 * n + 2 + n_exp_bits + n_ancilla))
    circ = Circuit(5 * n + 2 + n_exp_bits + n_ancilla)

    circ.x[x[0]]  # x starts at 1
    for i in range(n):
        if (val_N >> i) & 1:
            circ.x[N[i]]
    for k in range(n_exp_bits):
        if (exponent_val >> k) & 1:
            circ.x[j[k]]

    def apply_modulo_add(ctrl, constant, ops_source):
        nonlocal circ
        for i in range(n):
            const_val = (constant * (1 << i)) % val_N
            for jj in range(n):
                if (const_val >> jj) & 1:
                    circ.x[const[jj]]
            for controls, target in ops_source(c, const, res, N, t, n):
                circ += mcx(list(controls) + [ctrl, x[i]], target, ancillas)
            for jj in range(n):
                if (const_val >> jj) & 1:
                    circ.x[const[jj]]

    for k in range(n_exp_bits):
        a_pow = pow(mult_a, 1 << k, val_N)      # a^(2^k) mod N, controlled by bit k of the exponent
        a_pow_inv = pow(a_pow, -1, val_N)
        apply_modulo_add(j[k], a_pow, modulo_add_ops)
        for controls, target in controlled_swap_ops(j[k], x, res[:n]):
            circ += mcx(list(controls), target, ancillas)
        apply_modulo_add(j[k], a_pow_inv, lambda *args: list(reversed(modulo_add_ops(*args))))

    return circ, x

In [6]:
def run_modexp(mult_a, val_N, exponent_val, n, n_exp_bits):
    circuit, x = build_modexp(mult_a, val_N, exponent_val, n, n_exp_bits)
    total = circuit.n_qubits
    print(f"  [qubits={total}, gates={len(circuit.ops)}]")
    state = circuit.m[:].run(shots=1).most_common(1)[0][0]
    bits = [state[total - 1 - q] for q in x]
    return int("".join(reversed(bits)), 2)

for exponent in [0, 1, 2, 3]:
    result = run_modexp(2, 3, exponent, 2, 2)
    expected = pow(2, exponent, 3)
    print(f"2^{exponent} mod 3 = {result}  (expected {expected}, {'OK' if result == expected else 'MISMATCH'})")

  [qubits=18, gates=2479]


2^0 mod 3 = 1  (expected 1, OK)
  [qubits=18, gates=2480]


2^1 mod 3 = 2  (expected 2, OK)
  [qubits=18, gates=2480]


2^2 mod 3 = 1  (expected 1, OK)
  [qubits=18, gates=2481]


2^3 mod 3 = 2  (expected 2, OK)


All four exponents match Python's own `pow(2, j, 3)` exactly, confirming the chain of in-place controlled multiplications is computing $2^j \bmod 3$ correctly for every $j$ from $0$ to $3$ — precisely what the 2-qubit exponent register can hold. Each run took a little over a minute here (18 qubits, roughly 2,500 gates): about $4\times$ Part 3's single-multiplication cost, since this chains 2 exponent bits, each of which costs one multiply + swap + un-multiply, i.e. roughly $2\times$ a Part 3 multiplication — consistent with the gate-count-driven scaling discussed there.

## Where this connects back

This is the last new arithmetic circuit in the series. In the real algorithm:

1. The exponent register above isn't set classically with `X` gates — it's put into an equal superposition over every value with Hadamard gates.
2. This same modular-exponentiation circuit runs *once*, unchanged, computing $a^j \bmod N$ for every $j$ in that superposition simultaneously — this is where the exponential parallelism actually happens.
3. An inverse QFT on the exponent register, followed by measurement, gives (with high probability) an estimate of a multiple of $1/r$ where $r$ is the order of $a$ mod $N$.
4. Classical continued-fraction post-processing turns that estimate into a candidate $r$, which is checked and, if even, used to extract a factor of $N$ via $\gcd(a^{r/2} \pm 1, N)$.

Steps 3 and 4 are exactly what [400_shor.ipynb](400_shor.ipynb) already works through in detail (including a full worked example for $N=15$ and $N=21$) — this series' job was to make sure step 2, the part that's usually left as "and then you build a modular exponentiation circuit" in the theory, is something you've actually built and tested yourself, gate by gate, from `X`/`CX`/`CCX` on up.